In [ ]:
import os
import cv2
import json
import tensorflow as tf
import mediapipe as mp

# Print TensorFlow version
print(tf.__version__)

# Enable GPU memory growth (if GPU is available)
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
    print("GPU Available:", physical_devices)

# MediaPipe Face Detection Setup
mp_face_detection = mp.solutions.face_detection
face_detector = mp_face_detection.FaceDetection(model_selection=1, min_detection_confidence=0.95)

# Define the base path for images
base_path = "./train_sample_images/"

# Function to get filename without extension
def get_filename_only(file_path):
    return os.path.splitext(os.path.basename(file_path))[0]

# Process each image in the dataset
for folder in os.listdir(base_path):
    folder_path = os.path.join(base_path, folder)
    
    if not os.path.isdir(folder_path):
        continue  # Skip if not a directory

    print(f"Processing Folder: {folder_path}")
    faces_path = os.path.join(folder_path, 'faces')
    os.makedirs(faces_path, exist_ok=True)

    # Process each image in the folder
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            image_path = os.path.join(folder_path, filename)
            print(f"Processing {filename}...")

            # Read the image
            image = cv2.imread(image_path)
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            # Detect faces
            results = face_detector.process(image_rgb)

            if results.detections:
                print(f"Faces Detected: {len(results.detections)}")

                count = 0
                for detection in results.detections:
                    bbox = detection.location_data.relative_bounding_box
                    h, w, _ = image.shape

                    # Convert bounding box coordinates to absolute values
                    x1, y1 = int(bbox.xmin * w), int(bbox.ymin * h)
                    x2, y2 = int((bbox.xmin + bbox.width) * w), int((bbox.ymin + bbox.height) * h)

                    # Add margin to bounding box
                    margin_x, margin_y = int(bbox.width * w * 0.3), int(bbox.height * h * 0.3)
                    x1, y1 = max(x1 - margin_x, 0), max(y1 - margin_y, 0)
                    x2, y2 = min(x2 + margin_x, w), min(y2 + margin_y, h)

                    # Crop the detected face
                    cropped_face = image[y1:y2, x1:x2]

                    # Save cropped face
                    new_filename = f"{get_filename_only(filename)}-{count:02d}.png"
                    output_filepath = os.path.join(faces_path, new_filename)
                    cv2.imwrite(output_filepath, cropped_face)
                    print(f"Saved: {output_filepath}")

                    count += 1
            else:
                print("No faces detected in:", filename)

print("Face extraction complete!")